In [ ]:
%pip install -U ddgs
from ddgs import DDGS

In [ ]:
%pip install easyocr
import easyocr
reader = easyocr.Reader(['en'])

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os
import io
import requests
import time

from PIL import Image

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

### Drive File Access

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
PATH = '/content/gdrive/MyDrive/Scraped_Images'

In [ ]:
df_1 = pd.read_parquet("/content/gdrive/My Drive/Colab Notebooks/test images/Food Ingredients Data/train-00000-of-00003.parquet")
df_2 = pd.read_parquet("/content/gdrive/My Drive/Colab Notebooks/test images/Food Ingredients Data/train-00001-of-00003.parquet")
df_3 = pd.read_parquet("/content/gdrive/My Drive/Colab Notebooks/test images/Food Ingredients Data/train-00002-of-00003.parquet")
df = pd.concat([df_1, df_2, df_3], axis=0)
df.head()

### Local File Access

In [ ]:
PATH = './scraped-images'

In [ ]:
df_1 = pd.read_parquet("./food-ingredients/train-00000-of-00003.parquet")
df_2 = pd.read_parquet("./food-ingredients/train-00001-of-00003.parquet")
df_3 = pd.read_parquet("./food-ingredients/train-00002-of-00003.parquet")
df = pd.concat([df_1, df_2, df_3], axis=0)
df.head()

In [ ]:
ingredients = list(set(list(zip(df['ingredient'].tolist(), df['subcategory'].tolist(), df['category'].tolist()))))
ingredients

In [ ]:
queries = df['ingredient'].tolist()
queries = list(set(queries))
queries

In [ ]:
def has_text(image_bytes):
  img = Image.open(io.BytesIO(image_bytes))
  results = reader.readtext(np.array(img))
  return len(results) > 0

In [ ]:
def fetch_image_bytes(url):
    try:
        response = requests.get(url)
    except requests.RequestException:
        return None

    if response.status_code != 200:
        return None

    data = response.content
    if len(data) == 0:
        return None

    return data

In [ ]:
regions = ["us-en", "uk-en", "au-en", "ca-en", "in-en", "ie-en", "nz-en", "sg-en", "za-en", "ph-en",
             "my-en", "id-en", "de-de", "fr-fr", "es-es", "it-it", "nl-nl", "ch-de", "ch-fr", "ch-it",
             "at-de", "be-fr", "be-nl", "se-sv", "no-no", "dk-da", "fi-fi", "pl-pl", "cz-cs", "hu-hu",
             "ro-ro", "sk-sk", "sl-sl", "hr-hr", "bg-bg", "ee-et", "lv-lv", "lt-lt", "gr-el", "ua-uk",
             "cn-zh", "jp-jp", "kr-kr", "hk-tzh", "tw-tzh", "th-th", "vn-vi", "id-id", "my-ms", "ph-tl",
             "il-he", "xa-ar", "br-pt", "mx-es", "ar-es", "cl-es", "co-es", "pe-es", "ve-es", "pt-pt",
             "ca-fr", "xl-es", "ue-es", "wt-wt", "ct-ca", "xa-en"]

In [ ]:
page_nums = [i for i in range(5)]

In [ ]:
def scrape_images(query, ingredient):
  name = query 
  print(name)
  query += ' ' + ingredient[1] + ' raw uncooked' #-cooked -vector -prepared
  
  with open('./progress.txt', 'r') as f:
    results = f.read().split('\n')
    length = len(name)
    count = [int(i.split(' ')[-1]) for i in results if i[:length] == name] 
    count = max(count) if count else 0
    print(count)
    f.close()

  with open('./urls.txt', 'r') as f:
    downloaded = set(f.read().split('\n'))

  num_idx = 0
  reg_idx = 0
  dir_path = os.path.join(PATH, f'{ingredient[0]}_{ingredient[1]}_{ingredient[2]}')
  os.makedirs(dir_path, exist_ok=True)

  while count < 100:
    try:
      results = DDGS().images(query=query, max_results=100, page=page_nums[num_idx], region=regions[reg_idx])
    except Exception as e:
      num_idx += 1
      if num_idx >= 5:
        reg_idx += 1
        num_idx = 0
        if reg_idx >= len(regions):
          print(e)
          print('Regions exhausted')

          with open('./progress.txt', 'r') as f:
            results = f.read().split('\n')
            length = len(name)
            idx = [results.index(i) for i in results if i[:length] == name]
            idx = max(idx) if idx else -1
            if idx != -1:
              results[idx] = f'{name} {count}'
            else:
              results.append(f'{name} {count}')
            f.close()

          with open('./progress.txt', 'w') as f:
            for i in results:
              f.write(i + '\n')
            f.close()

          break
      continue

    for each in results:
      if count >= 100:
        break

      try:
        data = fetch_image_bytes(each['image'])
        
        if data is None:
          continue
        if has_text(data):
          continue
        if each['image'] in downloaded:
          continue
        
        path = os.path.join(dir_path, f"{name}_{count + 1}.jpg")
        with open(path, 'wb') as f:
          f.write(data)

        with open('./urls.txt', 'a+') as f:
          f.write(each['image'] + '\n')
          f.close()
        downloaded.add(each['image'])

        count += 1
      except Exception as e:
        print(e)
        print(f'Failed to download {name} at {count + 1}')
        continue

    num_idx += 1
    if num_idx >= 5:
        reg_idx += 1
        num_idx = 0
    time.sleep(10)

  with open('./progress.txt', 'r') as f:
    results = f.read().split('\n')
    length = len(name)
    idx = [results.index(i) for i in results if i[:length] == name]
    idx = max(idx) if idx else -1
    if idx != -1:
      results[idx] = f'{name} {count}'
    else:
      results.append(f'{name} {count}')
    f.close()

  with open('./progress.txt', 'w') as f:
    for i in results:
      f.write(i + '\n')
    f.close()

In [ ]:
try:
  scrape_images('spinach', ['spinach', 'leafy', 'vegetables']) #test query
except Exception as e:
  print(e)

In [ ]:
  
progress_file = './progress.txt'
if os.path.exists(progress_file):
  with open(progress_file, 'r') as f:
    lines = f.read()
else:
  lines = ''
done = lines.split('\n')
for item in queries:
  if f'{item} 100' in done:
    continue
  full_ingredient = [[ingredients[i][0], ingredients[i][1], ingredients[i][2]] for i in range(len(ingredients)) if ingredients[i][0] == item]
  # print(full_ingredient)
  try:
    scrape_images(item, full_ingredient[0])
  except Exception as e:
    print(e)
  time.sleep(30)